In [4]:
import feedparser
import requests
import pandas as pd
import os
from bs4 import BeautifulSoup
from slugify import slugify  # Install using pip install python-slugify

# Hacker News RSS feed URL
RSS_FEED_URL = "https://feeds.feedburner.com/TheHackersNews"

# Directory to save the blog text files
TEXT_FILES_DIR = "hacker_news_blogs_texts"
os.makedirs(TEXT_FILES_DIR, exist_ok=True)

# Function to fetch blog details from an individual blog URL and save text content
def fetch_blog_details(blog_url):
    try:
        response = requests.get(blog_url)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Try finding the title based on known structure
        title_element = soup.find('h1', class_='story-title')
        if not title_element:
            title_element = soup.find('title')  # Fallback to page title
        title = title_element.text.strip() if title_element else 'No title found'

        # Try finding the publication date based on known structure
        pub_date_element = soup.find('span', class_='story_date')
        pub_date = pub_date_element.text.strip() if pub_date_element else 'No date found'

        # Try finding the blog content/summary based on known structure
        content_element = soup.find('div', class_='articlebody clear cf')
        content = content_element.text.strip() if content_element else 'No content found'

        # Save content to a .txt file
        if content:
            file_name = f"{slugify(title)}.txt"
            file_path = os.path.join(TEXT_FILES_DIR, file_name)

            with open(file_path, "w", encoding="utf-8") as file:
                file.write(f"Title: {title}\n")
                file.write(f"Date: {pub_date}\n")
                file.write(f"URL: {blog_url}\n\n")
                file.write("Content:\n")
                file.write(content)

        return title, pub_date, content

    except Exception as e:
        print(f"Error fetching blog details from {blog_url}: {e}")
        return None, None, None

# Parse the RSS feed
def parse_rss_feed(rss_feed_url):
    feed = feedparser.parse(rss_feed_url)
    blogs = []

    # Loop over each blog entry in the feed
    for entry in feed.entries:
        blog_url = entry.link
        title, pub_date, content = fetch_blog_details(blog_url)

        if title and pub_date and content:
            blogs.append({
                'title': title,
                'pub_date': pub_date,
                'summary': content,
                'url': blog_url
            })

    return blogs

# Main function to execute the scraper
def scrape_hacker_news():
    print("Fetching blogs from The Hacker News...")
    blogs = parse_rss_feed(RSS_FEED_URL)

    # Convert the blogs data into a pandas DataFrame
    df = pd.DataFrame(blogs)
    
    # Save the DataFrame to a CSV file for further analysis or storage
    if not df.empty:
        df.to_csv("hacker_news_blogs.csv", index=False)
    else:
        print("No blogs found. Something went wrong.")

    print(f"Scraped {len(blogs)} blogs.")
    return df

# Run the scraper
if __name__ == "__main__":
    df = scrape_hacker_news()
    print(df.head())




Fetching blogs from The Hacker News...
Scraped 50 blogs.
                                               title       pub_date  \
0  North Korean Hackers Deploy FudModule Rootkit ...  No date found   
1  Cyberattackers Exploit Google Sheets for Malwa...  No date found   
2  Iranian Hackers Set Up New Network to Target U...  No date found   
3  Breaking Down AD CS Vulnerabilities: Insights ...  No date found   
4  New Malware Masquerades as Palo Alto VPN Targe...  No date found   

                                             summary  \
0  A recently patched security flaw in Google Chr...   
1  Cybersecurity researchers have uncovered a nov...   
2  Cybersecurity researchers have unearthed new n...   
3  The most dangerous vulnerability you've never ...   
4  Cybersecurity researchers have disclosed a new...   

                                                 url  
0  https://thehackernews.com/2024/08/north-korean...  
1  https://thehackernews.com/2024/08/cyberattacke...  
2  https://the